In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

data = [
  # IMDB
  {"dataset":"IMDB","model":"BERT","setting":"FT","f1":93.43,"cost":25.44,"latency_p50":480.06},
  {"dataset":"IMDB","model":"DistilBERT","setting":"FT","f1":92.73,"cost":12.44,"latency_p50":234.82},
  {"dataset":"IMDB","model":"RoBERTa","setting":"FT","f1":94.84,"cost":32.98,"latency_p50":622.21},
  {"dataset":"IMDB","model":"GPT-4o","setting":"ZS","f1":96.11,"cost":842.78,"latency_p50":344.89},
  {"dataset":"IMDB","model":"GPT-4o","setting":"FS","f1":96.16,"cost":1537.78,"latency_p50":394.84},
  {"dataset":"IMDB","model":"Claude 4.5","setting":"ZS","f1":96.45,"cost":1174.95,"latency_p50":1312.94},
  {"dataset":"IMDB","model":"Claude 4.5","setting":"FS","f1":96.48,"cost":2120.01,"latency_p50":1314.52},

  # SST-2
  {"dataset":"SST-2","model":"BERT","setting":"FT","f1":94.42,"cost":7.79,"latency_p50":147.03},
  {"dataset":"SST-2","model":"DistilBERT","setting":"FT","f1":93.49,"cost":5.19,"latency_p50":97.88},
  {"dataset":"SST-2","model":"RoBERTa","setting":"FT","f1":93.59,"cost":7.07,"latency_p50":133.38},
  {"dataset":"SST-2","model":"GPT-4o","setting":"ZS","f1":87.00,"cost":192.48,"latency_p50":377.10},
  {"dataset":"SST-2","model":"GPT-4o","setting":"FS","f1":90.45,"cost":387.48,"latency_p50":326.00},
  {"dataset":"SST-2","model":"Claude 4.5","setting":"ZS","f1":91.78,"cost":326.67,"latency_p50":1394.27},
  {"dataset":"SST-2","model":"Claude 4.5","setting":"FS","f1":94.41,"cost":599.67,"latency_p50":1109.11},

  # AG News
  {"dataset":"AG News","model":"BERT","setting":"FT","f1":94.43,"cost":10.44,"latency_p50":196.91},
  {"dataset":"AG News","model":"DistilBERT","setting":"FT","f1":94.11,"cost":5.73,"latency_p50":108.19},
  {"dataset":"AG News","model":"RoBERTa","setting":"FT","f1":94.63,"cost":10.00,"latency_p50":188.70},
  {"dataset":"AG News","model":"GPT-4o","setting":"ZS","f1":87.93,"cost":276.00,"latency_p50":410.81},
  {"dataset":"AG News","model":"GPT-4o","setting":"FS","f1":89.65,"cost":903.50,"latency_p50":332.31},
  {"dataset":"AG News","model":"Claude 4.5","setting":"ZS","f1":91.35,"cost":440.58,"latency_p50":1434.82},
  {"dataset":"AG News","model":"Claude 4.5","setting":"FS","f1":90.56,"cost":1271.58,"latency_p50":1002.56},

  # DBPedia
  {"dataset":"DBPedia","model":"BERT","setting":"FT","f1":99.40,"cost":10.77,"latency_p50":203.23},
  {"dataset":"DBPedia","model":"DistilBERT","setting":"FT","f1":99.40,"cost":7.03,"latency_p50":132.57},
  {"dataset":"DBPedia","model":"RoBERTa","setting":"FT","f1":99.33,"cost":10.90,"latency_p50":205.63},
  {"dataset":"DBPedia","model":"GPT-4o","setting":"ZS","f1":96.12,"cost":463.20,"latency_p50":406.32},
  {"dataset":"DBPedia","model":"GPT-4o","setting":"FS","f1":97.06,"cost":1903.20,"latency_p50":417.39},
  {"dataset":"DBPedia","model":"Claude 4.5","setting":"ZS","f1":98.83,"cost":751.89,"latency_p50":1127.83},
  {"dataset":"DBPedia","model":"Claude 4.5","setting":"FS","f1":98.39,"cost":2701.89,"latency_p50":1126.59},
]

df = pd.DataFrame(data)

MARKER = {"FT":"o", "ZS":"s", "FS":"^"}

MODEL_COLOR = {
    "BERT": "tab:blue",
    "DistilBERT": "tab:orange",
    "RoBERTa": "tab:green",
    "GPT-4o": "tab:red",
    "Claude 4.5": "tab:purple",
}

def pareto_mask(points, maximize_cols, minimize_cols):
    """
    Return boolean mask True for Pareto-optimal points.
    """
    X = points.copy()
    for c in maximize_cols:
        X[c] = -X[c]
    cols = minimize_cols + maximize_cols
    A = X[cols].to_numpy()
    n = A.shape[0]
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_pareto[i]:
            continue
        dominated = np.all(A <= A[i], axis=1) & np.any(A < A[i], axis=1)
        dominated[i] = False
        if np.any(dominated):
            is_pareto[i] = False
    return is_pareto

def plot_scatter(
    dataset, xcol, ycol, xlog=False, annotate_pareto=True,
    outpath="fig.png", xlabel=None, ylabel=None, title=None,
    label_all_points=True
):
    d = df[df["dataset"] == dataset].copy()

    fig = plt.figure(figsize=(4.2, 3.2), dpi=300)
    ax = plt.gca()

    # Scatter + label above each point
    for _, r in d.iterrows():
        ax.scatter(
            r[xcol], r[ycol],
            marker=MARKER[r["setting"]],
            s=60,
            color=MODEL_COLOR[r["model"]],
            edgecolors="black",
            linewidths=0.5,
            alpha=0.95
        )

        if label_all_points:
            ax.annotate(
                r["model"],
                (r[xcol], r[ycol]),
                textcoords="offset points",
                xytext=(0, 6),     # text above point
                ha="center",
                fontsize=7,
                color=MODEL_COLOR[r["model"]],
                alpha=0.95
            )

    if xlog:
        ax.set_xscale("log")

    if title:
        ax.set_title(title, fontsize=10)

    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.5)

    # Optional Pareto labels only (if you want to label only Pareto points set label_all_points=False)
    if annotate_pareto and (not label_all_points):
        if (xcol, ycol) == ("cost", "f1"):
            m = pareto_mask(d, maximize_cols=["f1"], minimize_cols=["cost"])
        elif (xcol, ycol) == ("cost", "latency_p50"):
            m = pareto_mask(d, maximize_cols=[], minimize_cols=["cost", "latency_p50"])
        elif (xcol, ycol) == ("latency_p50", "f1"):
            m = pareto_mask(d, maximize_cols=["f1"], minimize_cols=["latency_p50"])
        else:
            m = np.zeros(len(d), dtype=bool)

        pareto_pts = d[m]
        for _, r in pareto_pts.iterrows():
            ax.annotate(
                f'{r["model"]} {r["setting"]}',
                (r[xcol], r[ycol]),
                textcoords="offset points",
                xytext=(0, 6),
                ha="center",
                fontsize=7,
                color=MODEL_COLOR[r["model"]],
                alpha=0.95
            )

    # Setup legend only (FT / ZS / FS)
    setup_handles = [
        Line2D([0], [0], marker=MARKER[s], color='black', label=s,
               markerfacecolor='white', markeredgecolor='black',
               markersize=7, linewidth=0)
        for s in ["FT", "ZS", "FS"]
    ]

    ax.legend(
      handles=setup_handles,
      title="Setup",
      loc="center left",
      bbox_to_anchor=(1.02, 0.5),
      fontsize=7,
      title_fontsize=8,
      frameon=True
    )

    plt.tight_layout()
    plt.savefig(outpath, bbox_inches="tight", dpi=300)
    plt.close(fig)

datasets = ["IMDB", "SST-2", "AG News", "DBPedia"]

for ds in datasets:
    ds_slug = ds.replace(" ", "_").lower()

    plot_scatter(
        ds, xcol="cost", ycol="f1",
        xlog=True,
        outpath=f"fig_{ds_slug}_f1_vs_cost.png",
        title=ds,
        xlabel="Estimated cost (USD / 1M requests, log scale)",
        ylabel="Macro-F1 (%)",
        label_all_points=True
    )

    plot_scatter(
        ds, xcol="cost", ycol="latency_p50",
        xlog=True,
        outpath=f"fig_{ds_slug}_cost_vs_latency.png",
        title=ds,
        xlabel="Estimated cost (USD / 1M requests, log scale)",
        ylabel="Inference latency p50 (ms)",
        label_all_points=True
    )

    plot_scatter(
        ds, xcol="latency_p50", ycol="f1",
        xlog=False,
        outpath=f"fig_{ds_slug}_f1_vs_latency.png",
        title=ds,
        xlabel="Inference latency p50 (ms)",
        ylabel="Macro-F1 (%)",
        label_all_points=True
    )
